# 🚀 WhyBook: Export to LiteRT for Google Edge Gallery

This notebook bridges our Hugging Face models to **Google Edge Gallery** by converting the fine-tuned `gemma-4-e2b` PyTorch weights into a **LiteRT** (`.task` or `.tflite`) format. 

*Note: You mentioned "guff format" in your prompt, but for Google Edge Gallery and Android deployments, we need the **LiteRT** format! (We already exported the GGUF format in Phase 2 for desktop/llama.cpp inference).* 

### Workflow:
1. Pull the Base Model (`google/gemma-2b-it` or `e2b`) from Hugging Face.
2. Pull our trained LoRA adapters (`Stinger2311/whybook-gemma4-e2b-lora`).
3. Merge them into a standard 32-bit/16-bit PyTorch model.
4. Use Google's `ai-edge-torch` to convert the PyTorch model into a highly optimized LiteRT model for on-device Android inference.

In [ ]:
# 1. Install necessary libraries
!pip install -q transformers peft torch accelerate huggingface_hub
# Install Google's AI Edge Torch converter (requires specific PyTorch versions typically, check ODML docs for latest)
!pip install -q ai-edge-torch


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

# Optional: Login to HF if pulling private base models
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

### Step 1: Download Base Model & Merge LoRA
LiteRT conversion requires standard PyTorch weights. We'll merge our PEFT/LoRA adapters back into the base model's standard linear layers.

In [ ]:
BASE_MODEL = "google/gemma-2b-it" # Replace with exact e2b model id if different
LORA_MODEL = "Stinger2311/whybook-gemma4-e2b-lora"
MERGED_DIR = "./whybook-gemma-merged-pytorch"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float32 # LiteRT conversion often prefers fp32 as a starting point before quantization
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Loading and merging LoRA adapters...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL)
model = model.merge_and_unload() # Fuses the LoRA weights into the base weights

print("Saving merged PyTorch model to disk...")
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Merge complete! Ready for LiteRT conversion.")

### Step 2: Convert to LiteRT (TFLite)
Google provides `ai-edge-torch` to translate PyTorch computation graphs directly into LiteRT flatbuffers. 
*Note: Generative LLM conversion to LiteRT can be highly specific to the model architecture. Google's MediaPipe LLM Inference API also provides automated scripts for Gemma.*

In [ ]:
import ai_edge_torch

LITERT_OUTPUT_PATH = "whybook-gemma-e2b.tflite"

# Note: Full LLM conversion requires tracing the forward pass with dummy inputs.
# For Gemma specifically, Google often recommends using their pre-built conversion scripts 
# provided in the MediaPipe repository to handle KV-caching optimizations automatically.

# Example conceptual conversion (refer to Google AI Edge documentation for Gemma-specific exact tracing):
'''
dummy_input = torch.tensor([[tokenizer.bos_token_id]])
edge_model = ai_edge_torch.convert(model, (dummy_input,))
edge_model.export(LITERT_OUTPUT_PATH)
print(f"Successfully exported LiteRT model to {LITERT_OUTPUT_PATH}")
'''

print("To execute the final conversion to .task/.tflite format, please use the official Google MediaPipe Gemma conversion script against the `MERGED_DIR` generated above.")
print("Reference: https://ai.google.dev/edge/mediapipe/solutions/genai/llm_inference/android")
